# IBM-1 — materialize a model from the declaration

[IBM-1](https://github.com/JacobFV/IBM-1) declares a brain **once** — fields, topologies and
processes — and traces task-specific models out of that declaration on demand.

This notebook runs that. No weights and no GPU are needed for the first part: the
declaration is the artifact, and materializing against it costs nothing but a `pip install`.

The last cell downloads the fused implicit kernel, which is the one parameter set every
materialization draws from.


In [ ]:
!pip install -q git+https://github.com/JacobFV/IBM-1


## 1. Load the ontology

The registry validates on every load. It refuses a process that reads a variable nothing
writes, an anatomical system with no recorded source, a band a component cannot carry, and a
dangling selector. `seal=True, strict=True` makes those refusals raise instead of warn.


In [ ]:
import ibm

ibm.load_all(seal=True, strict=True)

# the whole ontology is designed to print as one table -- if it does not,
# it has already begun to sprawl (ARCHITECTURE.md §8)
print(ibm.REGISTRY.summary())


## 2. What can be materialized

Every entry differs only in the regions, resolution and bandwidth it asks for.


In [ ]:
from ibm.materialize import library

MODELS = library.MODELS
print(f"{len(MODELS)} declared models\n")
for mid, m in sorted(MODELS.items()):
    first = (m.doc or '').strip().splitlines()
    print(f"  {mid:26s} {first[0][:58] if first else ''}")


## 3. Materialize one

Naming a target is the whole interface. Dependency tracing decides what has to exist;
the rest of the brain is never built.

Try swapping `eeg_forward` for `tms_response`, `sleep_dynamics`, `eeg_to_image` or
`invasive_bci`.


In [ ]:
m = MODELS["eeg_forward"]

print(f"== {m.id} ==")
print((m.doc or '').strip()[:400])
print()
print(m.request.describe())


### The provenance travels with it

A materialized model carries what constrains it and what does not. `prior_dominated` is the
honest half: those components will come out smooth because nothing in the listed sources
distinguishes them.


In [ ]:
print(f"fit sources   {', '.join(m.fit_sources) or '(none)'}")
print(f"eval sources  {', '.join(m.eval_sources) or '(none)'}")
print(f"controls      {', '.join(m.negative_controls) or '(none)'}")
print()
print('constrained by the listed sources:')
for c in m.constrained:
    print(f'  + {c}')
print('prior-dominated -- smooth, because nothing here distinguishes it:')
for c in m.prior_dominated:
    print(f'  ~ {c}')


## 4. The implicit kernel

One association kernel over the cortical sheet, no heads. Every materialization this
programme has trained is that kernel plus a small task head, and the heads are the cheap
part — so you download the kernel, not five models.

Each kernel ships a sidecar carrying what it was measured at, **including the caveats**.
Read the `note` field before quoting any of these numbers.


In [ ]:
import json, torch
from huggingface_hub import hf_hub_download

REPO = "jacob-valdez/ibm-1"
STEM = "implicit/ibm1.implicit.s30k.e128.fused34"

pt   = hf_hub_download(REPO, STEM + ".pt")
side = json.load(open(hf_hub_download(REPO, STEM + ".json")))

d = torch.load(pt, map_location="cpu")
assert d["schema"] == "ibm1/implicit-v1", d.get("schema")
e = d["dyn.embed"]

print(side["what_it_is"])
print()
print(f"embedding   {tuple(e.shape)}  ->  {e.shape[0] * e.shape[1]:,} parameters")
print(f"fused from  {side['n_sources']} checkpoints")
print()
for k, v in side["measured"].items():
    print(f"  {k:42s} {v}")


### Materializing a task model from it

The kernel resamples to whatever resolution you ask for, by k-NN on the sphere. Sites are
drawn from a seeded stream whose point sets differ between resolutions, so asking for a
resolution it was not fused at is an approximation — and it is **reported** as one rather
than performed silently.

```bash
python scripts/materialize.py --target visual_eeg --sites 30000
```

---

Every withdrawn claim in this programme, with the check that caught it, is in
[`docs/LOG.md`](https://github.com/JacobFV/IBM-1/blob/main/docs/LOG.md). Checkpoints are
published for runs that *failed* too — a negative result without a checkpoint is an anecdote.
